In [ ]:
# !conda install -y -c conda-forge imbalanced-learn

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
csv_files = []
for dirname, _, filenames in os.walk('../input/cicids2017/MachineLearningCSV/MachineLearningCVE'):
    for filename in filenames:
        csv_file = os.path.join(dirname, filename)
        print(os.path.join(dirname, filename))
        csv_files.append(csv_file)

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
# pd.read_csv(csv_files[0])

In [ ]:
df = pd.concat([pd.read_csv(file) for file in csv_files], ignore_index=True)
df.columns = df.columns.str.strip()
print("original length of df:", len(df))
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.dropna(inplace=True)
print("after droping null values, the length of df:", len(df))
# df_experiment = df.loc[df.Label.str.contains(pat='DoS|BENIGN',na=False)]
# df_experiment = df_experiment.loc[~(df_experiment.Label =="DoS Slowhttptest") ]

# del df


In [ ]:
df.Label.value_counts()
# del df

In [ ]:
df_experiment = df.copy()
df_experiment.Label.replace("Web.*", "Web Attack", regex=True, inplace=True)
df_experiment.Label.replace(r'.*Patator$', "Brute Force", regex=True,inplace=True)
df_experiment.Label.replace(["DoS GoldenEye", "DoS Hulk", "DoS Slowhttptest", "DoS slowloris"], "DoS", inplace=True)
df_experiment.Label.value_counts()

In [ ]:
df_experiment.head(2)

In [ ]:
# df_experiment.Label.value_counts()
del df

In [ ]:
# data split
from sklearn.model_selection import train_test_split
from collections import Counter
y = df_experiment.Label
X = df_experiment.drop(columns='Label')
labels = y.unique()

classes = y.nunique()
print(X.shape)
print("number of labels:", classes)
print("instances per label\n", y.value_counts())
print("labels:", labels)

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42, stratify=y)
# print(len(df_experiment))
# print(df_experiment.Label.nunique())
# print(df_experiment.Label.value_counts())
print("after spliting the data:\n")
print("training data length:", len(X_train))
print("test data length:", len(X_test))
# print(y_train.nunique())
# print(y_train.value_counts())

In [ ]:
### Set a copy for binary classification:
class_attack = ['PortScan', 'Web Attack', 'Brute Force', 'DDoS', 'Bot','Infiltration', 'DoS', 'Heartbleed']
bi_train_y = y_train.copy()
bi_train_y.replace(class_attack, value='attack', inplace=True)
print(bi_train_y.unique())
# bi_train.head(2)

bi_test_y = y_test.copy()
bi_test_y.replace(class_attack, value='attack', inplace=True)
print(bi_test_y.unique())
# bi_train.head(2)

### switch for binary!
y_train = bi_train_y
y_test = bi_test_y

In [ ]:
import seaborn as sns
sns.countplot(x=y_train);

In [ ]:
from sklearn.preprocessing import OneHotEncoder, LabelEncoder, MinMaxScaler
# enc = OneHotEncoder(handle_unknown='ignore')
scaler = MinMaxScaler()
le = LabelEncoder()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)
print("instances per label in training set\n", y_train.value_counts())
y_train = le.fit_transform(y_train)
print("instances per label in test set\n", y_test.value_counts())
y_test = le.transform(y_test)

print(X_train.shape)
print(X_test.shape)
labels_dict = dict(zip(le.classes_, range(len(le.classes_))))
print(labels_dict)


### training data sampling
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline
from collections import Counter
over = SMOTE(sampling_strategy='not majority', n_jobs=-1)
under = RandomUnderSampler(sampling_strategy=
                             {labels_dict['BENIGN']:200000})
# steps = [('u', under), ('o', over)]
steps = [('o', over)]
pipeline = Pipeline(steps=steps)
X_train, y_train = pipeline.fit_resample(X_train, y_train)
counter = Counter(y_train)
print(counter)

In [ ]:
X.shape

In [ ]:
X.info()

In [ ]:
X.describe()

In [ ]:
# Autoencoder model to extract latent features

# from tensorflow.keras import Model, Sequential, Input
# from tensorflow.keras.layers import Dense
# from tensorflow.keras.callbacks import EarlyStopping
# callback = EarlyStopping(patience=3, mode='min')

# input_shape = X_train.shape[1]
# print(f"input shape: {input_shape}")
# encoding_dim = 10
# encoder = Sequential([
#     Dense(32, activation='relu', input_shape=(input_shape, )),
#     Dense(encoding_dim, activation='relu')
# ])
# decoder = Sequential([
#     Dense(32, activation='relu', input_shape=(encoding_dim,)),
#     Dense(input_shape)
# ])
# autoencoder = Sequential([encoder, decoder])
# autoencoder.compile(loss='mse', optimizer='Adam')
# history = autoencoder.fit(X_train, X_train, batch_size=64,
#                           epochs=20, validation_split=0.2)

In [ ]:
# X_train_encode = encoder.predict(X_train)
# X_test_encode = encoder.predict(X_test)

In [ ]:
# from sklearn.decomposition import PCA
# import matplotlib.pyplot as plt
# import seaborn as sns
# from sklearn.utils import resample
# %matplotlib inline

# X_train_pca, y_train_pca = resample(
#     X_train, y_train, n_samples=30000, random_state=42, stratify=y_train)
# pca = PCA(n_components=2)
# projected = pca.fit_transform(X_train_pca)
# fig, ax = plt.subplots(figsize=(10,6))
# scat = ax.scatter(projected[:, 0], projected[:, 1],
#             c=y_train_pca, edgecolor='none', cmap="Paired", alpha=0.7)
# plt.legend(handles=scat.legend_elements()[0], labels=list(le.classes_), loc="upper left", title="Classes")
# plt.axis('off')
# plt.title("PCA at latent space of CIC-IDS2017 training set")
# plt.show()

In [ ]:
# X_test_pca, y_test_pca = resample(
#     X_test, y_test, n_samples=100000, random_state=42, stratify=y_test)
# pca = PCA(n_components=2)
# projected = pca.fit_transform(X_test_pca)
# fig, ax = plt.subplots(figsize=(10,6))
# scat = ax.scatter(projected[:, 0], projected[:, 1],
#             c=y_test_pca, edgecolor='none', cmap="Paired", alpha=0.7)
# plt.legend(handles=scat.legend_elements()[0], labels=list(le.classes_), loc="upper left", title="Classes")
# plt.axis('off')
# plt.title("PCA at latent space of CIC-IDS2017 test set")
# plt.show()

## Feature selection:

### ANOVA f-value

In [ ]:
### transform ndarray into dataframe type for later operation.
X_train = pd.DataFrame(X_train)
X_test = pd.DataFrame(X_test)

from sklearn.feature_selection import f_classif
### Compute the ANOVA F-value 
f_value = f_classif(X_train, y_train)

In [ ]:
# plot the importance score
import seaborn as sns
import matplotlib.pyplot as plt

f_value_series = pd.Series(f_value[0])
f_value_series.index = X.columns

df_importance = f_value_series.sort_values(ascending=False)
importance_scores, feature_names = df_importance.values, df_importance.index

sns.set(style="whitegrid")
#设置子图的大小
f, ax = plt.subplots(figsize=(6, 20))
sns.set_color_codes("muted")

sns.barplot(x = importance_scores, y = feature_names)

#设置坐标信息
ax.set(xlim=(0, max(importance_scores)), ylabel="", xlabel="importance score", title="ANOVA Feature Importance")
sns.despine(left=True, bottom=True)
plt.show()

In [ ]:
### Now we select the top 10 important features based on the result above.
from sklearn.feature_selection import SelectKBest
sel_top_cols = SelectKBest(f_classif, k=10)
sel_top_cols.fit(X_train, y_train)
X_train = X_train[X_train.columns[sel_top_cols.get_support()]]
X_test = X_test[X_test.columns[sel_top_cols.get_support()]]

### transform ndarray back:
X_train = X_train.values
X_test = X_test.values

In [ ]:
# print("X_train.shape: ", X_train.shape)
# print("X_test.shape: ", X_test.shape)

## Classification Model---LSTM 

In [ ]:
### reshape input data to LSTM format [samples, time_steps, features]
X_train_lstm = X_train.reshape(X_train.shape[0], 1, X_train.shape[1])
X_test_lstm = X_test.reshape(X_test.shape[0], 1, X_test.shape[1])
print(f"shape of X_train:", X_train_lstm.shape)
print(f"shape of X_test:", X_test_lstm.shape)

In [ ]:
from tensorflow.keras import Model, Sequential, Input, backend
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

n_classes = len(le.classes_)
print(f"num of classes:{n_classes}")
n_features = X_train_lstm.shape[2]
def multiClassModel(n_features, n_classes=9):
    model = Sequential()
    model.add(Input(shape=(None, n_features)))
    model.add(LSTM(units=30))
    model.add(Dropout(0.2))
    model.add(Dense(n_classes, activation="softmax", name="softmax"))
    model.compile(loss="sparse_categorical_crossentropy", optimizer='Adam')
    model.summary()
    return model

In [ ]:
# from keras.callbacks import EarlyStopping
callback = EarlyStopping(patience=20, mode='min', restore_best_weights=True)
backend.clear_session()
model = multiClassModel(n_features, n_classes)
history = model.fit(X_train_lstm, y_train, 
                    epochs=200, batch_size=64, validation_split=0.2, callbacks=[callback])
### check the loss trend of epochs
pd.DataFrame(history.history).plot(kind='line', xlabel='epochs', figsize=(8, 6))

import matplotlib.pyplot as plt
plt.show()

In [ ]:
# predicting on training set
y_train_pred_prob = model.predict(X_train_lstm)
y_test_pred_prob = model.predict(X_test_lstm)
y_train_pred = np.argmax(y_train_pred_prob, axis=1)
y_test_pred = np.argmax(y_test_pred_prob, axis=1)

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline
cm = confusion_matrix(y_train, y_train_pred)
fig, ax = plt.subplots(figsize=(12, 12))
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=le.classes_).plot(ax=ax)
plt.show()

In [ ]:
cm = confusion_matrix(y_test, y_test_pred)
fig, ax = plt.subplots(figsize=(12, 12))
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=le.classes_).plot(ax=ax)
plt.show()

In [ ]:
def multilabel_matrix(y_true, y_pred, labels=None):
    mlm = multilabel_confusion_matrix(y_true, y_pred, labels=labels)
    df_performance = pd.DataFrame(index=labels, columns=['accuracy', 'precision', 'recall', 'f1_score'])
    for i, label in enumerate(labels):
        tn, fp, fn, tp = mlm[i].ravel()
        accuracy = (tn + tp) / (tn + fp + fn + tp)
        precision = tp / (tp + fp)
        recall = tp / (tp + fn)

        f1_score = 2*precision * recall / (precision + recall)
        df_performance.loc[label] = [round(accuracy, 4), round(precision,4), \
                                     round(recall, 4), round(f1_score,4)]
    return df_performance

In [ ]:
from tensorflow.keras.utils import to_categorical
from sklearn.metrics import roc_curve, auc
from sklearn.metrics import roc_auc_score
from itertools import cycle
def RoC_Curve(y_score, y, labels, title): 
    y_cat = to_categorical(y)
    
    # Compute ROC curve and ROC area for each class
    fpr = dict()
    tpr = dict()
    roc_auc = dict()
    lw = 2
    # First aggregate all false positive rates
    n_classes = len(labels)
#     print('n_classes:', n_classes)

    for i in range(n_classes):
        fpr[i], tpr[i], _ = roc_curve(y_cat[:, i], y_score[:, i])
        roc_auc[i] = auc(fpr[i], tpr[i])

    # Compute micro-average ROC curve and ROC area
    fpr["micro"], tpr["micro"], _ = roc_curve(y_cat.ravel(), y_score.ravel())
    roc_auc["micro"] = auc(fpr["micro"], tpr["micro"])

    # First aggregate all false positive rates
    all_fpr = np.unique(np.concatenate([fpr[i] for i in range(n_classes)]))

    # Then interpolate all ROC curves at this points
    mean_tpr = np.zeros_like(all_fpr)
    for i in range(n_classes):
        mean_tpr += np.interp(all_fpr, fpr[i], tpr[i])

    # Finally average it and compute AUC
    mean_tpr /= n_classes

    fpr["macro"] = all_fpr
    tpr["macro"] = mean_tpr
    roc_auc["macro"] = auc(fpr["macro"], tpr["macro"])

    # Plot all ROC curves
    plt.figure(figsize=(8,8))
    plt.plot(fpr["micro"], tpr["micro"],
             label='micro-average ROC curve (area = {0:0.4f})'
                   ''.format(roc_auc["micro"]),
             color='deeppink', linestyle=':', linewidth=4)

    plt.plot(fpr["macro"], tpr["macro"],
             label='macro-average ROC curve (area = {0:0.4f})'
                   ''.format(roc_auc["macro"]),
             color='navy', linestyle=':', linewidth=4)

    for i in range(n_classes):
        plt.plot(fpr[i], tpr[i], lw=lw,
                 label=f'ROC curve of class {labels[i]} (area = {roc_auc[i]:0.4f})')

    plt.plot([0, 1], [0, 1], 'k--', lw=lw)
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(title, fontsize=16)
    plt.legend(loc="lower right")
    plt.show()

In [ ]:
from sklearn.metrics import classification_report, multilabel_confusion_matrix
y_train_pred_labels = le.inverse_transform(y_train_pred)
y_train_labels = le.inverse_transform(y_train)
print(classification_report(y_train_labels, y_train_pred_labels))
performance = multilabel_matrix(y_train_pred_labels, y_train_labels, labels=le.classes_)
performance

In [ ]:
y_test_pred_labels = le.inverse_transform(y_test_pred)
y_test_true_labels = le.inverse_transform(y_test)
print(classification_report(y_test_true_labels,y_test_pred_labels))

In [ ]:
performance = multilabel_matrix(y_test_true_labels, y_test_pred_labels, labels=le.classes_)
performance

In [ ]:
RoC_Curve(y_train_pred_prob, y_train, le.classes_, title='ROC for CIC-IDS2017 training set')

In [ ]:
RoC_Curve(y_test_pred_prob, y_test, le.classes_, title='ROC for CIC-IDS2017 test set')